In [1]:
import random
import uuid
from datetime import datetime, timedelta
import pandas as pd
from faker import Faker
from sqlalchemy import create_engine, inspect
import sqlite3

fake = Faker()

In [2]:
def simulate_data(start_date, end_date, n_users,transitions,initial_state):
    # Convert input dates
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")

    def choose_next(state,transitions):
        next_states, probs = zip(*transitions[state])
        return random.choices(next_states, probs, k=1)[0]

    users = []
    events = []

    for _ in range(n_users):
        user_id = str(uuid.uuid4())
        first_visit = fake.date_time_between(start_date=start_date, end_date=end_date)

        user = {
            "id": user_id,
            "first_visit": first_visit,
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "sex": random.choice(["F", "M"]),
            "age": random.randint(18, 62),
            "platform": random.choice(["web", "mobile"]),
            "zone": random.choice(["North", "South", "West", "East"])
        }
        users.append(user)

        # Generación de eventos
        session_count = 0
        current_state = initial_state
        current_time = first_visit

        while session_count < 8:
            events.append({
                "id": str(uuid.uuid4()),
                "user_id": user_id,
                "event_date": current_time,
                "event_name": current_state
            })

            next_state = choose_next(current_state,transitions)

            # Loop → terminar
            if next_state == current_state:
                break

            # Si el usuario vuelve al initial_state → sumar sesión y usar retorno largo
            if next_state == initial_state:
                session_count += 1
                if session_count >= 10:
                    break
                # Tiempo entre 18 y 200 horas (retorno natural)
                current_time += timedelta(hours=random.uniform(18, 200))
            else:
                # Tiempo entre eventos normales: 18 a 900 segundos
                current_time += timedelta(seconds=random.randint(18, 900))

            current_state = next_state

    users_df = pd.DataFrame(users)
    events_df = pd.DataFrame(events).sort_values(by=["user_id", "event_date"])

    return users_df, events_df



## User Journey db

In [ ]:
transitions = {
        "sesion_start": [
            ("sesion_start", 0.20),
            ("search_page", 0.80)
        ],
        "search_page": [
            ("search_page", 0.05),
            ("sesion_start", 0.27),
            ("product_page", 0.68),
        ],
        "product_page": [
            ("product_page", 0.15),
            ("car_checkout", 0.58),
            ("sesion_start", 0.27),
        ],
        "car_checkout": [
            ("car_checkout", 0.16),
            ("payment_confirmation", 0.57),
            ("sesion_start", 0.27),
        ],
        "payment_confirmation": [
            ("payment_confirmation", 0.35),
            ("sesion_start", 0.65),
        ]
    }
initial_state='sesion_start'

users, events = simulate_data("2025-01-01", "2025-04-30", 2131,transitions,initial_state)


 # ========================
# 2️⃣ Crear conexión SQLite nueva base
# ========================
sqlite_filename = "user_journey.db"
sqlite_conn = sqlite3.connect(sqlite_filename)

for table,df in {'events':events,'users':users}.items():
        df.to_sql(table, sqlite_conn, if_exists='replace', index=False)
        print(f"✅ Tabla '{table}' copiada con {len(df)} registros.")

## Journey DB

In [6]:
initial_state='session_start'

transitions = {
        "session_start": [
            ("session_start", 0.05),
            ("view_item", 0.95)
        ],
        "view_item": [
            ("view_item", 0.05),
            ("session_start", 0.17),
            ("add_to_cart", 0.78),
        ],
        "add_to_cart": [
            ("add_to_cart", 0.15),
            ("begin_checkout", 0.68),
            ("session_start", 0.17),
        ],
        "begin_checkout": [
            ("begin_checkout", 0.09),
            ("purchase", 0.70),
            ("session_start", 0.21),
        ],
        "purchase": [
            ("purchase", 0.45),
            ("session_start", 0.55),
        ]
    }

users, events = simulate_data("2021-01-01", "2021-04-30", 5248,transitions,initial_state)


 # ========================
# 2️⃣ Crear conexión SQLite nueva base
# ========================
sqlite_filename = "journeys.db"
sqlite_conn = sqlite3.connect(sqlite_filename)

for table,df in {'events':events,'users':users}.items():
        df.to_sql(table, sqlite_conn, if_exists='replace', index=False)
        print(f"✅ Tabla '{table}' copiada con {len(df)} registros.")

✅ Tabla 'events' copiada con 44933 registros.
✅ Tabla 'users' copiada con 5248 registros.


✅ Tabla 'events' copiada con 13634 registros.
✅ Tabla 'users' copiada con 2131 registros.
